# Análisis GAC · Etapa I. Modelado explicativo (Análisis comparativo)
## a) Extracción de características – Análisis univariado

**Bases de datos:**
- `Analisis_GAC_Full7agosto_2026_Bitacora_de_Piso_.csv` → base limpia: **`Bitacora_Piso_Limpia.csv`**
- `Analisis_GAC_Full7agosto_2026_Leads_Reales_.csv` → base limpia: **`Leads_Reales_Limpia.csv`**

**Variables analizadas (no se repiten Estatus Lead, Producto ni Interés de leads):**

| Base | Variable | Tipo de análisis | Razón |
|---|---|---|---|
| Bitácora de Piso | `Asesor` | Categórica (frecuencia de visitas atendidas) | Mide cómo se reparte la carga de clientes de piso entre asesores |
| Bitácora de Piso | `Prueba_Manejo` (PDM 0/1) | Binaria numérica (Sí / No) | Primer paso del embudo de venta en piso; indica qué tanto se convierte una visita en prueba |
| Leads Reales | `Nivel_Efectividad` (a partir de `Efectividad_Leads` %) | Numérica transformada a categórica (Baja / Media / Alta) | Clasifica cada mes según la calidad de los leads recibidos |

## Instalación de librerías

In [ ]:
#Instalamos la libreria de PLOTLY
%pip install plotly

---
## Preprocesamiento – Bitácora de Piso
### Lectura y exploración inicial

In [ ]:
#Importamos librerias
import pandas as pd
import numpy as np

In [ ]:
#Lectura del archivo csv (utf-8-sig elimina el caracter raro del inicio del archivo)
df_bitacora = pd.read_csv("Analisis_GAC_Full7agosto_2026_Bitacora_de_Piso_.csv", encoding="utf-8-sig")
#Revisamos dimensiones y primeras filas
print(df_bitacora.shape)
df_bitacora.head()

In [ ]:
#Revisamos los nombres de las columnas (varios traen espacios al final)
print(df_bitacora.columns.tolist())

In [ ]:
#Conteo de nulos por columna
df_bitacora.isnull().sum()

### Limpieza de nombres de columnas y columnas vacías
- Se quitan los espacios sobrantes de los nombres de las columnas.
- Se corrige el nombre `Esatus Lead` → `Estatus_Lead` y se renombran columnas para no usar espacios.
- Se eliminan las columnas `Unnamed` (catálogos auxiliares de Excel, no son registros) y `Telefono` (100% nula).

In [ ]:
#Quitamos espacios al inicio y final de los nombres de columnas
df_bitacora.columns = df_bitacora.columns.str.strip()

#Renombramos columnas para trabajar sin espacios
df_bitacora = df_bitacora.rename(columns={'Nombre Cliente': 'Cliente',
                                          'Esatus Lead': 'Estatus_Lead',
                                          'Estatus de SDC': 'Estatus_SDC',
                                          'Inter Gerente': 'Inter_Gerente'})

#Eliminamos columnas "Unnamed" (catálogos de Excel) y Telefono (vacía)
Columnas_Unnamed = df_bitacora.columns[df_bitacora.columns.str.contains('Unnamed')]
df_bitacora = df_bitacora.drop(columns=Columnas_Unnamed)
df_bitacora = df_bitacora.drop(columns=['Telefono'])
print(df_bitacora.shape)

### Tratamiento de nulos
**Filas vacías:** el archivo trae renglones en blanco al final. Se eliminan las filas que no tienen `Cliente`, `Asesor` **ni** `Estatus_Lead` (no representan ninguna visita real).

In [ ]:
#Eliminamos filas que no tienen cliente, asesor ni estatus (renglones vacíos)
df_bitacora = df_bitacora.dropna(subset=['Cliente', 'Asesor', 'Estatus_Lead'], how='all')
print(df_bitacora.shape)
df_bitacora.isnull().sum()

**Columnas con más del 90% de nulos:** `Estatus_SDC`, `Temperatura` y las columnas de texto abierto (¿Por qué no...?, 5 porqués, Comentarios) casi no tienen información. Imputarlas inventaría datos, por lo que se eliminan del análisis univariado.

In [ ]:
#Calculamos el porcentaje de nulos por columna
Porcentaje_Nulos = df_bitacora.isnull().mean() * 100
print(Porcentaje_Nulos.round(1))

#Eliminamos las columnas con más de 90% de nulos
Columnas_Eliminar = Porcentaje_Nulos[Porcentaje_Nulos > 90].index
df_bitacora = df_bitacora.drop(columns=Columnas_Eliminar)
print("Columnas eliminadas:", Columnas_Eliminar.tolist())

**Imputación de los nulos restantes:** quedan muy pocos nulos en variables categóricas.
- `Asesor` nulo → `"No identificado"` (no se puede saber quién atendió).
- `Cliente` nulo → `"Sin nombre"`.
- `Estatus_Lead` nulo → se imputa con la **moda**.

In [ ]:
#Imputamos nulos en variables categóricas
df_bitacora['Asesor'] = df_bitacora['Asesor'].fillna('No identificado')
df_bitacora['Cliente'] = df_bitacora['Cliente'].fillna('Sin nombre')
df_bitacora['Estatus_Lead'] = df_bitacora['Estatus_Lead'].fillna(df_bitacora['Estatus_Lead'].mode()[0])

#Verificamos que ya no existan nulos
df_bitacora.isnull().sum()

### Transformaciones
- **Texto:** se quitan espacios y se homologan mayúsculas (`venta`, `Venta ` → `Venta`).
- **Asesor:** un mismo asesor aparece escrito de varias formas (`Aurelio`, `Aurelio Torres`, `AUrelio`). Se toma solo el primer nombre en formato título.
- **Fecha:** se convierte a tipo fecha (formato mes/día/año) y se crean `Mes_Num` y `Mes`.
- **Booleanos:** `PDM`, `SDC`, `Venta`, `Inter_Gerente` pasan de TRUE/FALSE a **1/0** (variables numéricas).
- Se crea la etiqueta `Prueba_Manejo` (Sí / No) a partir de `PDM` para los gráficos.

In [ ]:
#Homologamos el texto de Estatus_Lead
df_bitacora['Estatus_Lead'] = df_bitacora['Estatus_Lead'].str.strip().str.title()
print(df_bitacora['Estatus_Lead'].value_counts())

In [ ]:
#Homologamos Asesor: quitamos espacios, tomamos el primer nombre y lo ponemos en formato título
df_bitacora['Asesor'] = df_bitacora['Asesor'].str.strip().str.split().str[0].str.title()
#"No identificado" queda como "No" al partir el texto, lo regresamos a su valor
df_bitacora['Asesor'] = df_bitacora['Asesor'].replace('No', 'No identificado')
print(df_bitacora['Asesor'].value_counts())

In [ ]:
#Convertimos la fecha (viene en formato mes/día/año)
df_bitacora['Fecha'] = pd.to_datetime(df_bitacora['Fecha'], format='%m/%d/%Y', errors='coerce')

#Creamos el número y nombre del mes
df_bitacora['Mes_Num'] = df_bitacora['Fecha'].dt.month
Nombres_Meses = {1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril', 5: 'Mayo', 6: 'Junio',
                 7: 'Julio', 8: 'Agosto', 9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'}
df_bitacora['Mes'] = df_bitacora['Mes_Num'].map(Nombres_Meses)
print(df_bitacora['Mes'].value_counts())

In [ ]:
#Convertimos las columnas TRUE/FALSE a numéricas 1/0
for columna in ['PDM', 'SDC', 'Venta', 'Inter_Gerente']:
    df_bitacora[columna] = df_bitacora[columna].astype(str).str.strip().str.upper().map({'TRUE': 1, 'FALSE': 0})

#Creamos la etiqueta de Prueba de Manejo para los gráficos
df_bitacora['Prueba_Manejo'] = df_bitacora['PDM'].map({1: 'Sí', 0: 'No'})
df_bitacora[['PDM', 'SDC', 'Venta', 'Inter_Gerente']].describe()

### Outliers
**Variable numérica – visitas por día:** se cuenta cuántas visitas se registran por día y se revisa con el método del **rango intercuartílico (IQR)**. Los días con muchas visitas son reales (fines de semana, eventos), por lo que **solo se reportan** y no se eliminan filas (cada fila es un cliente real).

**Variable categórica – Asesor:** en variables categóricas los "outliers" son categorías con muy pocos registros. Los asesores con **menos de 5 visitas** se agrupan en la categoría **`Otros`** para que no distorsionen los gráficos.

In [ ]:
#Contamos visitas por día
Visitas_Dia = df_bitacora.groupby('Fecha').size()

#Calculamos cuartiles y límites del IQR
Q1 = Visitas_Dia.quantile(0.25)
Q3 = Visitas_Dia.quantile(0.75)
IQR = Q3 - Q1
Limite_Inferior = Q1 - 1.5 * IQR
Limite_Superior = Q3 + 1.5 * IQR
print("Límite inferior:", Limite_Inferior, "| Límite superior:", Limite_Superior)

#Mostramos los días fuera de los límites
Outliers_Dia = Visitas_Dia[(Visitas_Dia < Limite_Inferior) | (Visitas_Dia > Limite_Superior)]
print("Días atípicos:")
print(Outliers_Dia)

In [ ]:
#Agrupamos asesores con menos de 5 visitas en "Otros"
Frecuencia_Asesor = df_bitacora['Asesor'].value_counts()
Asesores_Poco_Frecuentes = Frecuencia_Asesor[Frecuencia_Asesor < 5].index
df_bitacora['Asesor'] = df_bitacora['Asesor'].replace(Asesores_Poco_Frecuentes, 'Otros')
print(df_bitacora['Asesor'].value_counts())

### Base limpia final y guardado

In [ ]:
#Ordenamos las columnas finales
df_bitacora = df_bitacora[['Fecha', 'Mes_Num', 'Mes', 'Cliente', 'Asesor', 'Estatus_Lead',
                           'PDM', 'SDC', 'Venta', 'Inter_Gerente', 'Prueba_Manejo']]
print(df_bitacora.shape)
print(df_bitacora.isnull().sum().sum(), "nulos restantes")

#Guardamos la base limpia
df_bitacora.to_csv("Bitacora_Piso_Limpia.csv", index=False)
df_bitacora.head()

---
## Preprocesamiento – Leads Reales
### Lectura y exploración inicial
El archivo es un reporte de Excel: las dos primeras filas son títulos/totales y debajo de la tabla hay cálculos de objetivos. Se leen **solo las filas de la tabla mensual** (`skiprows=2`, `nrows=30`).

In [ ]:
#Lectura del archivo csv (solo la tabla mensual)
df_leads = pd.read_csv("Analisis_GAC_Full7agosto_2026_Leads_Reales_.csv", encoding="utf-8-sig",
                       skiprows=2, nrows=30)
print(df_leads.shape)
df_leads

In [ ]:
#Conteo de nulos por columna
df_leads.isnull().sum()

### Limpieza de columnas
- Se quitan espacios en los nombres y se renombran sin espacios ni puntos.
- Se eliminan las columnas `Unnamed` (vacías).

In [ ]:
#Quitamos espacios de los nombres de columnas
df_leads.columns = df_leads.columns.str.strip()

#Renombramos columnas
df_leads = df_leads.rename(columns={'No Contesta': 'No_Contesta',
                                    'D. Incorrectos': 'D_Incorrectos',
                                    'Efectividad de Leads': 'Efectividad_Leads'})

#Eliminamos columnas vacías "Unnamed"
Columnas_Unnamed = df_leads.columns[df_leads.columns.str.contains('Unnamed')]
df_leads = df_leads.drop(columns=Columnas_Unnamed)
df_leads.columns.tolist()

### Tratamiento de nulos
- **`Año`** solo aparece en el primer mes de cada año (celdas combinadas en Excel) → se rellena hacia abajo con **`ffill()`**.
- Se elimina la fila vacía del final (sin `Mes`).
- **Enero 2024** no tiene ningún dato registrado → se elimina (imputarla inventaría un mes completo de operación).

In [ ]:
#Eliminamos la fila sin mes
df_leads = df_leads.dropna(subset=['Mes'])

#Rellenamos el año hacia abajo
df_leads['Año'] = df_leads['Año'].ffill().astype(int)

#Eliminamos el mes sin registros (Enero 2024)
df_leads = df_leads.dropna(subset=['Total'])
print(df_leads.shape)
df_leads.isnull().sum()

### Transformaciones
- `Mes` trae espacios al final (`"Enero "`) → se limpian.
- Las cantidades vienen como texto con comas (`"1,409"`) → se convierten a número.
- `Conversion` y `Efectividad_Leads` vienen como texto con `%` → se convierten a número (porcentaje).
- Se crean `Mes_Num` y `Periodo` (Año-Mes) para ordenar en el tiempo.

In [ ]:
#Limpiamos espacios en el mes
df_leads['Mes'] = df_leads['Mes'].str.strip()

#Convertimos cantidades con coma a números
Columnas_Cantidad = ['Total', 'Ilocalizable', 'No_Contesta', 'D_Incorrectos', 'Duplicados',
                     'Efectivos', 'Abiertos', 'Ventas']
for columna in Columnas_Cantidad:
    df_leads[columna] = df_leads[columna].astype(str).str.replace(',', '').str.strip().astype(float)

#Convertimos porcentajes a número
for columna in ['Conversion', 'Efectividad_Leads']:
    df_leads[columna] = df_leads[columna].astype(str).str.replace('%', '').str.strip().astype(float)

#Creamos el número de mes y el periodo
Numeros_Meses = {'Enero': 1, 'Febrero': 2, 'Marzo': 3, 'Abril': 4, 'Mayo': 5, 'Junio': 6,
                 'Julio': 7, 'Agosto': 8, 'Septiembre': 9, 'Octubre': 10, 'Noviembre': 11, 'Diciembre': 12}
df_leads['Mes_Num'] = df_leads['Mes'].map(Numeros_Meses)
df_leads['Periodo'] = df_leads['Año'].astype(str) + '-' + df_leads['Mes_Num'].astype(str).str.zfill(2)
df_leads.dtypes

### Outliers (método IQR)
Se revisa cada variable numérica con el rango intercuartílico. Como la base es mensual (28 registros), **no se eliminan filas** (se perderían meses completos); los valores fuera de rango se **acotan (winsorización)** al límite inferior o superior con `clip()`.

In [ ]:
#Revisamos outliers con IQR en cada variable numérica
Columnas_Numericas = Columnas_Cantidad + ['Conversion', 'Efectividad_Leads']
for columna in Columnas_Numericas:
    Q1 = df_leads[columna].quantile(0.25)
    Q3 = df_leads[columna].quantile(0.75)
    IQR = Q3 - Q1
    Limite_Inferior = Q1 - 1.5 * IQR
    Limite_Superior = Q3 + 1.5 * IQR
    Num_Outliers = ((df_leads[columna] < Limite_Inferior) | (df_leads[columna] > Limite_Superior)).sum()
    print(columna, "-> outliers:", Num_Outliers, "| límites:", round(Limite_Inferior, 2), "a", round(Limite_Superior, 2))
    #Acotamos los valores atípicos a los límites
    df_leads[columna] = df_leads[columna].clip(lower=Limite_Inferior, upper=Limite_Superior).round(2)

In [ ]:
#Boxplot rápido para verificar después del tratamiento
df_leads[['Efectividad_Leads', 'Conversion']].plot(kind='box', color='#800020', title='Después del tratamiento de outliers')

### Transformación de variable numérica a categórica: `Nivel_Efectividad`
Para el análisis univariado categórico, `Efectividad_Leads` se discretiza con `pd.cut`:
- **Baja:** ≤ 55%
- **Media:** 55% – 70%
- **Alta:** > 70%

In [ ]:
#Discretizamos la efectividad en niveles
df_leads['Nivel_Efectividad'] = pd.cut(df_leads['Efectividad_Leads'], bins=[0, 55, 70, 100],
                                       labels=['Baja', 'Media', 'Alta'])
df_leads['Nivel_Efectividad'].value_counts()

### Base limpia final y guardado

In [ ]:
#Ordenamos columnas finales
df_leads = df_leads[['Año', 'Mes_Num', 'Mes', 'Periodo', 'Total', 'Ilocalizable', 'No_Contesta',
                     'D_Incorrectos', 'Duplicados', 'Efectivos', 'Abiertos', 'Ventas',
                     'Conversion', 'Efectividad_Leads', 'Nivel_Efectividad']]
print(df_leads.shape)
print(df_leads.isnull().sum().sum(), "nulos restantes")

#Guardamos la base limpia
df_leads.to_csv("Leads_Reales_Limpia.csv", index=False)
df_leads.head()

---
## Análisis univariado – Gráficos

In [ ]:
#Importamos librerias para las gráficas
import plotly.express as px
import plotly.graph_objects as go

#Paleta de colores vino tinto, borgoña y blanco
Paleta_Vino = ['#4A0414', '#800020', '#9B1B30', '#B03A48', '#C9747F', '#DDA0A8', '#6D2E46', '#EFD3D7']
Colores_Nivel = {'Alta': '#4A0414', 'Media': '#9B1B30', 'Baja': '#DDA0A8'}

### Asesor – Gráfico de Pareto
Muestra cuántas visitas atendió cada asesor (barras) y el porcentaje acumulado (línea), para ver qué tan concentrada está la atención del piso.

In [ ]:
#Tabla de frecuencias de Asesor
Tabla_Asesor = df_bitacora['Asesor'].value_counts().reset_index()
Tabla_Asesor.columns = ['categorias', 'frecuencia']

#Porcentaje y porcentaje acumulado
Tabla_Asesor['porcentaje'] = (Tabla_Asesor['frecuencia'] / Tabla_Asesor['frecuencia'].sum() * 100).round(1)
Tabla_Asesor['porcentaje_acumulado'] = Tabla_Asesor['porcentaje'].cumsum().round(1)

#Gráfico de Pareto: barras de frecuencia + línea de % acumulado en un segundo eje
figure1 = go.Figure()
figure1.add_trace(go.Bar(x=Tabla_Asesor['categorias'], y=Tabla_Asesor['frecuencia'],
                         name='Frecuencia', marker_color='#800020', text=Tabla_Asesor['frecuencia']))
figure1.add_trace(go.Scatter(x=Tabla_Asesor['categorias'], y=Tabla_Asesor['porcentaje_acumulado'],
                             name='% acumulado', yaxis='y2', mode='lines+markers',
                             line=dict(color='#C9747F', width=3)))
figure1.update_layout(title='Pareto de visitas por asesor', height=420, plot_bgcolor='#FFFFFF',
                      yaxis=dict(title='Visitas'),
                      yaxis2=dict(overlaying='y', side='right', range=[0, 105], title='% acumulado'))
figure1.show()

**Interpretación:** la atención del piso está concentrada. Aurelio atiende cerca de una cuarta parte de las visitas y, junto con Pedro y Mauricio, los tres primeros asesores suman alrededor del 62% del total.

### Prueba_Manejo – Gráfico de dona
Muestra la proporción de visitas que sí y no realizaron prueba de manejo, el primer paso del embudo de venta en piso.

In [ ]:
#Tabla de frecuencias de Prueba_Manejo
Tabla_PDM = df_bitacora['Prueba_Manejo'].value_counts().reset_index()
Tabla_PDM.columns = ['categorias', 'frecuencia']

#Gráfico de dona
figure2 = px.pie(data_frame=Tabla_PDM, names='categorias', values='frecuencia', hole=0.45,
                 title='Visitas con y sin prueba de manejo', color='categorias',
                 color_discrete_map={'Sí': '#800020', 'No': '#DDA0A8'})
#Mostramos porcentaje y etiqueta, con borde blanco entre secciones
figure2.update_traces(textinfo='percent+label', marker=dict(line=dict(color='#FFFFFF', width=2)))
figure2.update_layout(height=420)
figure2.show()

**Interpretación:** solo alrededor del 38% de las visitas realiza prueba de manejo. La mayoría de los clientes se va sin probar el vehículo, lo que limita el avance hacia la solicitud de crédito y la venta.

### Nivel_Efectividad – Gráfico de barras en el tiempo
Muestra la efectividad de leads de cada mes, coloreada según su nivel, con líneas de corte en 55% y 70%.

In [ ]:
#Gráfico de barras por periodo, coloreado por nivel de efectividad
figure3 = px.bar(data_frame=df_leads, x='Periodo', y='Efectividad_Leads', color='Nivel_Efectividad',
                 color_discrete_map=Colores_Nivel, title='Efectividad de leads por mes',
                 category_orders={'Nivel_Efectividad': ['Baja', 'Media', 'Alta']})

#Líneas de corte de los niveles
figure3.add_hline(y=55, line_dash='dash', line_color='#9B1B30', annotation_text='55%')
figure3.add_hline(y=70, line_dash='dash', line_color='#4A0414', annotation_text='70%')
figure3.update_layout(height=420, plot_bgcolor='#FFFFFF', yaxis_title='Efectividad (%)')
figure3.show()

**Interpretación:** la mayoría de los meses (14 de 28) tiene efectividad Media, 8 son Baja y solo 6 son Alta. El mejor mes fue febrero 2025 (83%) y el peor agosto 2024 (39%); desde finales de 2025 la efectividad tiende a mantenerse en nivel Bajo o Medio.